Importing the dataset from  Kaggle

In [ ]:
from google.colab import files
import os

print("Please upload your Kaggle API JSON file:")
uploaded = files.upload()

# Automatically grab the exact name of the file you uploaded (e.g., 'kaggle (2).json')
filename = list(uploaded.keys())[0]

# Create the hidden kaggle directory
!mkdir -p ~/.kaggle

# Move the uploaded file and force it to be named 'kaggle.json'
os.rename(filename, '/root/.kaggle/kaggle.json')

# Secure the file permissions
!chmod 600 /root/.kaggle/kaggle.json
print("SUCCESS! Kaggle API Key setup complete!")

Please upload your Kaggle API JSON file:


Saving kaggle (2).json to kaggle (2) (1).json
SUCCESS! Kaggle API Key setup complete!


In [ ]:
!kaggle datasets download -d datamunge/sign-language-mnist
!unzip -q sign-language-mnist.zip -d sign_mnist_dataset

Dataset URL: https://www.kaggle.com/datasets/datamunge/sign-language-mnist
License(s): CC0-1.0
100% 62.6M/62.6M [00:00<00:00, 202MB/s]



Importing data

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import os

# Locate the CSV files
train_path, test_path = '', ''
for dirname, _, filenames in os.walk('/content/sign_mnist_dataset'):
    for filename in filenames:
        if filename == 'sign_mnist_train.csv': train_path = os.path.join(dirname, filename)
        elif filename == 'sign_mnist_test.csv': test_path = os.path.join(dirname, filename)

# Load the data
train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

# Separate labels and normalize pixel data (0 to 1)
y_train = train_df['label'].values
X_train = train_df.drop('label', axis=1).values.reshape(-1, 1, 28, 28) / 255.0

y_test = test_df['label'].values
X_test = test_df.drop('label', axis=1).values.reshape(-1, 1, 28, 28) / 255.0

print(f"Data Loaded! Training Images: {X_train.shape[0]}")

Data Loaded! Training Images: 27455


Creating the data Loaders

In [ ]:
class ImageSignDataset(Dataset):
    def __init__(self, images, labels):
        self.images = torch.tensor(images, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self): return len(self.labels)
    def __getitem__(self, idx): return self.images[idx], self.labels[idx]

# Create DataLoaders
train_loader = DataLoader(ImageSignDataset(X_train, y_train), batch_size=128, shuffle=True)
test_loader = DataLoader(ImageSignDataset(X_test, y_test), batch_size=128, shuffle=False)

print("DataLoaders are ready!")

DataLoaders are ready!


The CNN architecture

In [ ]:
class SignLanguageCNN(nn.Module):
    def __init__(self):
        super(SignLanguageCNN, self).__init__()
        # Convolutions extract visual features
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3, padding=1)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.conv2 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.3) # Prevents overfitting

        # Linear layers make the final guess (26 classes for A-Z)
        self.fc1 = nn.Linear(64 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, 26)

    def forward(self, x):
        x = self.pool1(self.relu(self.conv1(x)))
        x = self.pool2(self.relu(self.conv2(x)))
        x = x.view(x.size(0), -1) # Flatten the 2D image to 1D
        x = self.dropout(self.relu(self.fc1(x)))
        x = self.fc2(x)
        return x

model = SignLanguageCNN()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

print(f"CNN Model built and pushed to: {device}")

CNN Model built and pushed to: cpu


Model training

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
num_epochs = 15

print("Starting training...")

for epoch in range(num_epochs):
    model.train()
    train_correct, train_loss = 0, 0.0

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * inputs.size(0)
        _, preds = torch.max(outputs, 1)
        train_correct += torch.sum(preds == labels.data)

    train_acc = train_correct.double() / len(train_loader.dataset)

    # Test Phase (Checking accuracy on unseen data)
    model.eval()
    test_correct = 0
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            test_correct += torch.sum(preds == labels.data)

    test_acc = test_correct.double() / len(test_loader.dataset)

    print(f'Epoch {epoch+1:02}/{num_epochs} | Train Acc: {train_acc:.4f} | Test Acc: {test_acc:.4f}')
    print("Training Compleded!")

Starting training...
Epoch 01/15 | Train Acc: 0.4351 | Test Acc: 0.7616
Epoch 02/15 | Train Acc: 0.8091 | Test Acc: 0.8592
Epoch 03/15 | Train Acc: 0.8986 | Test Acc: 0.9013
Epoch 04/15 | Train Acc: 0.9408 | Test Acc: 0.8970
Epoch 05/15 | Train Acc: 0.9611 | Test Acc: 0.9142
Epoch 06/15 | Train Acc: 0.9708 | Test Acc: 0.8954
Epoch 07/15 | Train Acc: 0.9769 | Test Acc: 0.9105
Epoch 08/15 | Train Acc: 0.9774 | Test Acc: 0.9189
Epoch 09/15 | Train Acc: 0.9831 | Test Acc: 0.9215
Epoch 10/15 | Train Acc: 0.9828 | Test Acc: 0.9170
Epoch 11/15 | Train Acc: 0.9850 | Test Acc: 0.9101
Epoch 12/15 | Train Acc: 0.9855 | Test Acc: 0.9074
Epoch 13/15 | Train Acc: 0.9871 | Test Acc: 0.9166
Epoch 14/15 | Train Acc: 0.9876 | Test Acc: 0.9179
Epoch 15/15 | Train Acc: 0.9903 | Test Acc: 0.9219


Saving the model

In [ ]:
# Save the final model weights
torch.save(model.state_dict(), 'sign_mnist_cnn.pth')
print("Model saved successfully! You can now download 'sign_mnist_cnn.pth' from the Colab files panel.")

Model saved successfully! You can now download 'sign_mnist_cnn.pth' from the Colab files panel.
